In [19]:
# ============================================================
# CELL 0 — full drop-in replacement
# ============================================================
"""
merge_mycotoxin_datasets.py
============================
Merges SDD2 (Alltech) and SDD1 (EFSA) emerging mycotoxin datasets into a single
harmonised CSV for binary detection modelling.

Scope: CZ, DK, NL, DE | harvest years 2019–2023
Target: detected = 1 if value > LOD, else 0 (non-detects coded as 0)
"""

import pandas as pd
import numpy as np
import re

SDD2_PATH   = "ews_cereals_myco_with_drivers_oogstjaar_obv_start_emerging_2026-05-29.csv"
SDD1_PATH   = "Wouter emerging mycotoxin data v2.csv"
OUTPUT_PATH = "merged_mycotoxins.csv"
WEATHER_PATH = "production regions per country crop and year with weather data_2026-07-16.csv"

COUNTRIES   = {"cz", "dk", "nl", "de"}
YEAR_MIN, YEAR_MAX = 2019, 2023

ALPHA3_TO_2 = {"cze": "cz", "dnk": "dk", "nld": "nl", "deu": "de"}


# ── 1. LOAD ────────────────────────────────────────────────────────────────────
sdd2 = pd.read_csv(SDD2_PATH, sep=";", low_memory=False)
sdd1 = pd.read_csv(SDD1_PATH, sep=",", low_memory=False)
weather = pd.read_csv(WEATHER_PATH, sep=",", low_memory=False)
weather.columns = [c.strip() for c in weather.columns]

print(f"Loaded  SDD2: {len(sdd2):,} rows   SDD1: {len(sdd1):,} rows")


def normalise_weather_col(col):
    m = re.match(r"^(month|week) (\d+) (.+)$", col)
    if m:
        return f"{m.group(1)}_{m.group(2)}_{m.group(3)}"
    return col

weather.columns = [normalise_weather_col(c) for c in weather.columns]


# -- REPAIR: domain-plausible ranges + generalized digit-grouping fix ----------
# Some weather values (relative_humidity, temperature, precipitation - both
# mean and std) arrive corrupted with thousands-style dot grouping applied
# across the whole number including its decimal digits, e.g. "78.3875" ->
# "783.875", or "-3.238" -> "-32.38"-type distortions. We reconstruct the true
# value by trying plausible integer-part lengths (1-3 digits) and picking the
# one that lands inside the domain-plausible range for that variable type.

def expected_range(col):
    if "relative_humidity" in col and "std" not in col:
        return (0, 100)
    if "relative_humidity" in col and "std" in col:
        return (0, 50)
    if "temperature" in col and "std" not in col:
        return (-40, 50)
    if "temperature" in col and "std" in col:
        return (0, 15)
    if "precipitation" in col and "std" not in col:
        return (0, 300)
    if "precipitation" in col and "std" in col:
        return (0, 100)
    return None


def repair_digit_grouping(series, rng):
    lo, hi = rng

    def fix_value(v):
        if pd.isna(v):
            return v
        s = str(v).strip()
        neg = s.startswith("-")
        s_clean = s.lstrip("-")
        digits = s_clean.replace(".", "")
        if digits == "" or not digits.isdigit():
            return pd.to_numeric(s, errors="coerce")

        for int_len in (1, 2, 3):
            if len(digits) > int_len:
                candidate_str = digits[:int_len] + "." + digits[int_len:]
                try:
                    val = float(candidate_str)
                except ValueError:
                    continue
                if neg:
                    val = -val
                if lo <= val <= hi:
                    return val

        try:
            val = float(s)
            return val
        except ValueError:
            return np.nan

    return series.apply(fix_value)


def repair_all_weather_cols(data, label):
    weather_cols = [c for c in data.columns if c.startswith("month_") or c.startswith("week_")]
    print(f"\nRepairing weather columns in {label}:")
    n_fixed_cols = 0
    for col in weather_cols:
        rng = expected_range(col)
        if rng is None:
            continue
        before_bad = pd.to_numeric(data[col], errors="coerce")
        before_bad_n = ((before_bad < rng[0]) | (before_bad > rng[1])).sum()
        if before_bad_n == 0:
            continue
        data[col] = repair_digit_grouping(data[col], rng)
        after_bad_n = ((data[col] < rng[0]) | (data[col] > rng[1])).sum()
        print(f"  {col}: {before_bad_n} corrupted -> {after_bad_n} still out of range")
        n_fixed_cols += 1
    print(f"  Total columns touched: {n_fixed_cols}")
    return data


weather = repair_all_weather_cols(weather, "weather file")


# -- 2. RENAME -----------------------------------------------------------------
SDD2_RENAME = {
    "sample_number": "sample_id",
    "country_code3": "samplecountry",
    "harvestyear":   "harvest_year",
    "concentration": "value",
}

SDD1_RENAME = {
    "sample_core_id":  "sample_id",
    "sampcountry":     "samplecountry",
    "harvestyear":     "harvest_year",
    "param":           "contaminant",
    "productname":     "product",
    "productcategory": "product_use",
    "resval":          "value",
    "reslod":          "lod",
    "resloq":          "loq",
}

sdd2 = sdd2.rename(columns=SDD2_RENAME)
sdd1 = sdd1.rename(columns=SDD1_RENAME)

sdd2 = repair_all_weather_cols(sdd2, "SDD2")

sdd2 = sdd2.loc[:, ~sdd2.columns.duplicated(keep="first")]

Loaded  SDD2: 5,831 rows   SDD1: 13,238 rows

Repairing weather columns in weather file:
  Total columns touched: 0

Repairing weather columns in SDD2:
  month_1_temperature_max_mean: 2030 corrupted -> 0 still out of range
  month_2_temperature_max_mean: 2285 corrupted -> 0 still out of range
  month_3_temperature_max_mean: 2643 corrupted -> 0 still out of range
  month_4_temperature_max_mean: 2458 corrupted -> 0 still out of range
  month_1_temperature_min_mean: 456 corrupted -> 0 still out of range
  month_2_temperature_min_mean: 656 corrupted -> 0 still out of range
  month_3_temperature_min_mean: 1254 corrupted -> 0 still out of range
  month_4_temperature_min_mean: 1038 corrupted -> 0 still out of range
  month_1_temperature_avg_mean: 779 corrupted -> 0 still out of range
  month_2_temperature_avg_mean: 878 corrupted -> 0 still out of range
  month_3_temperature_avg_mean: 1933 corrupted -> 0 still out of range
  month_4_temperature_avg_mean: 1836 corrupted -> 0 still out of range


In [20]:
# -- CHECK: is analytical method already in the raw data? ---------------------
print("--- SDD2 (Alltech): method_analysis ---")
print(sdd2["method_analysis"].value_counts(dropna=False))

print("\n--- SDD2 (Alltech): method_analysis by contaminant (enniatins only) ---")
enniatin_all = ["enniatin a", "enniatin a1", "enniatin a + a1",
                "enniatin b", "enniatin b1", "enniatin b + b1"]
print(
    sdd2[sdd2["contaminant"].isin(enniatin_all)]
    .groupby(["contaminant", "method_analysis"])
    .size()
    .unstack(fill_value=0)
)

print("\n--- SDD1 (EFSA): restype ---")
print(sdd1["restype"].value_counts(dropna=False))

print("\n--- SDD1 (EFSA): restypecode ---")
print(sdd1["restypecode"].value_counts(dropna=False))

print("\n--- SDD1 (EFSA): restype by contaminant (enniatins only) ---")
print(
    sdd1[sdd1["contaminant"].isin(enniatin_all)]
    .groupby(["contaminant", "restype"])
    .size()
    .unstack(fill_value=0)
)

--- SDD2 (Alltech): method_analysis ---
method_analysis
lc-ms/ms    5831
Name: count, dtype: int64

--- SDD2 (Alltech): method_analysis by contaminant (enniatins only) ---
method_analysis  lc-ms/ms
contaminant              
enniatin a + a1       568
enniatin b + b1       568

--- SDD1 (EFSA): restype ---
restype
non quantified value ( less than loq)    9618
numerical value                          2103
non detected value ( less than lod)      1517
Name: count, dtype: int64

--- SDD1 (EFSA): restypecode ---
restypecode
loq    9618
val    2103
lod    1517
Name: count, dtype: int64

--- SDD1 (EFSA): restype by contaminant (enniatins only) ---
restype      non detected value ( less than lod)  \
contaminant                                        
enniatin a                                    58   
enniatin a1                                   53   
enniatin b                                    48   
enniatin b1                                   49   

restype      non quantified value ( les

In [4]:
print("SDD1 columns:")
print(sdd1.columns.tolist())

print("\nSDD2 columns:")
print(sdd2.columns.tolist())

print("\nWeather columns:")
print(weather.columns.tolist())

SDD1 columns:
['sample_id', 'samplehash', 'sampid', 'sampanid', 'progid', 'proglegalref', 'progtype', 'labaccred', 'labcountry', 'labsubsampcode', 'sampmethod', 'sampstrategy', 'anportseq', 'anportsizeunit', 'samppoint', 'samplecountry', 'origcountry', 'sampy', 'sampm', 'sampd', 'product', 'analysisy', 'analysism', 'analysisd', 'filename', 'country_code', 'filetype', 'resid', 'contaminant', 'paramcode', 'restype', 'restypecode', 'resunit', 'value', 'loq', 'evalcode', 'measurement_nr', 'file_id', 'product_id', 'evallowlimit', 'reslegallimit', 'legallimit', 'param_id', 'lod', 'product_use', 'productgroup', 'origcountry_id', 'code3', 'sampcountry_id', 'production_tons_x', 'harvest_year', 'year', 'corruption_index', 'human_development_index', 'control_corruption', 'government_effectiveness', 'political_stability', 'regulatory_quality', 'rule_of_law', 'voice_and_accountability', 'governance_index', 'press_freedom_index', 'democracy_index', 'polity_democracy_index', 'gdp_current', 'gdp_growt

In [5]:
# SDD1 arrives with month_*/week_* temperature, precipitation and
# relative_humidity columns already merged in from its original source.
# We drop these here and re-attach weather from WEATHER_PATH below instead.
# SDD2's own weather columns are left untouched, since weather is only linked
# to SDD1.
WEATHER_COL_PREFIXES = ("month_", "week_")
WEATHER_COL_KEYWORDS = ("temperature", "precipitation", "relative_humidity")
 
def drop_prelinked_weather(df):
    drop_cols = [
        c for c in df.columns
        if c.startswith(WEATHER_COL_PREFIXES) and any(k in c for k in WEATHER_COL_KEYWORDS)
    ]
    return df.drop(columns=drop_cols)
 
sdd1 = drop_prelinked_weather(sdd1)


In [6]:
print("SDD1 columns:")
print(sdd1.columns.tolist())

SDD1 columns:
['sample_id', 'samplehash', 'sampid', 'sampanid', 'progid', 'proglegalref', 'progtype', 'labaccred', 'labcountry', 'labsubsampcode', 'sampmethod', 'sampstrategy', 'anportseq', 'anportsizeunit', 'samppoint', 'samplecountry', 'origcountry', 'sampy', 'sampm', 'sampd', 'product', 'analysisy', 'analysism', 'analysisd', 'filename', 'country_code', 'filetype', 'resid', 'contaminant', 'paramcode', 'restype', 'restypecode', 'resunit', 'value', 'loq', 'evalcode', 'measurement_nr', 'file_id', 'product_id', 'evallowlimit', 'reslegallimit', 'legallimit', 'param_id', 'lod', 'product_use', 'productgroup', 'origcountry_id', 'code3', 'sampcountry_id', 'production_tons_x', 'harvest_year', 'year', 'corruption_index', 'human_development_index', 'control_corruption', 'government_effectiveness', 'political_stability', 'regulatory_quality', 'rule_of_law', 'voice_and_accountability', 'governance_index', 'press_freedom_index', 'democracy_index', 'polity_democracy_index', 'gdp_current', 'gdp_growt

In [7]:
# -- 3. HARMONISE SDD2 ---------------------------------------------------------
sdd2["samplecountry"] = sdd2["samplecountry"].str.lower().map(ALPHA3_TO_2)
sdd2["harvest_year"]  = pd.to_numeric(sdd2["harvest_year"], errors="coerce")
sdd2["value"]         = pd.to_numeric(sdd2["value"], errors="coerce")
sdd2["lod"]           = pd.to_numeric(sdd2["lod"], errors="coerce")
 
sdd2 = sdd2.copy().assign(
    source="alltech",
    detected=np.where(
        sdd2["lod"].notna(),
        (sdd2["value"] > sdd2["lod"]).astype("int8"),
        (sdd2["value"] > 0).astype("int8"),
    )
)



In [8]:
# -- 4. HARMONISE SDD1 ---------------------------------------------------------
SDD1_COUNTRY_MAP = {
    "czechia":      "cz",
    "denmark":      "dk",
    "netherlands":  "nl",
    "germany":      "de",
}

sdd1["samplecountry"] = sdd1["samplecountry"].str.lower().map(SDD1_COUNTRY_MAP)
sdd1["harvest_year"]  = pd.to_numeric(sdd1["harvest_year"], errors="coerce")
sdd1["value"]         = pd.to_numeric(sdd1["value"], errors="coerce")
sdd1["lod"]           = pd.to_numeric(sdd1["lod"], errors="coerce")
 
sdd1 = sdd1.copy().assign(
    source="efsa",
    detected=np.where(
        sdd1["lod"].notna(),
        (sdd1["value"].fillna(0) > sdd1["lod"]).astype("int8"),
        (sdd1["value"].fillna(0) > 0).astype("int8"),
    )
)



In [9]:
print("SDD2 samplecountry unique:", sdd2["samplecountry"].unique()[:10])
print("SDD2 harvest_year unique:", sdd2["harvest_year"].unique()[:10])
print()
print("SDD1 samplecountry unique:", sdd1["samplecountry"].unique()[:10])
print("SDD1 harvest_year unique:", sdd1["harvest_year"].unique()[:10])

SDD2 samplecountry unique: <StringArray>
['dk', 'de', 'cz', 'nl']
Length: 4, dtype: str
SDD2 harvest_year unique: [2019. 2020. 2021. 2022. 2023. 2024. 2014. 2017. 2015. 2018.]

SDD1 samplecountry unique: <StringArray>
[nan, 'cz', 'de', 'dk', 'nl']
Length: 5, dtype: str
SDD1 harvest_year unique: [2022. 2023. 2019. 2018. 2020. 2021.   nan]


In [10]:
# -- 5. SCOPE FILTER -----------------------------------------------------------
def scope_filter(df):
    return df[
        df["samplecountry"].isin(COUNTRIES) &
        df["harvest_year"].between(YEAR_MIN, YEAR_MAX)
    ].copy()
 
sdd2 = scope_filter(sdd2)
sdd1 = scope_filter(sdd1)
 
print(f"After filter  SDD2: {len(sdd2):,} rows   SDD1: {len(sdd1):,} rows")
 
 

 
 



After filter  SDD2: 5,170 rows   SDD1: 9,357 rows


In [11]:
# -- 6. COLUMN SELECTION -------------------------------------------------------
CORE_COLS = [
    "sample_id", "source", "samplecountry", "harvest_year",
    "contaminant", "product", "product_harmonised", "product_use",
    "value", "lod", "loq", "detected", "detected2", "detected3",
]
SDD2_EXTRA = [
    "organic", "gmo", "monitoring_type",
    "pre_crop_name", "pre_crop_percentage",
    "pre_pre_crop_name", "pre_pre_crop_percentage",
    "tillage", "transgenic_hybrid", "variety_group",
    "resistance", "fungicide", "irrigation",
    "agricultural_practices", "days_harvest_to_drying",
    "flowering_date", "harvest_date", "wgs84n", "wgs84e",
]
SDD1_EXTRA = [
    "progtype", "labaccred", "samppoint", "sampmethod", "sampstrategy",
    "origcountry", "sampm", "sampd", "evalcode",
    "restype", "reslegallimit", "legallimit",
]
 
exclude = set(CORE_COLS) | set(SDD2_EXTRA) | set(SDD1_EXTRA)
sdd2_drivers = [c for c in sdd2.columns if c not in exclude]
sdd1_drivers = [c for c in sdd1.columns if c not in exclude]
 
def select_cols(df, core, extra, drivers):
    cols = [c for c in core + extra + drivers if c in df.columns]
    return df[cols].copy()
 
sdd2 = select_cols(sdd2, CORE_COLS, SDD2_EXTRA, sdd2_drivers)
sdd1 = select_cols(sdd1, CORE_COLS, SDD1_EXTRA, sdd1_drivers)

In [12]:
# -- 6b. LOQ INVESTIGATION & NORMALISED DETECTION COLUMNS ---------------------

# Inspect LOQ distribution across both datasets combined (pre-merge)
loq_all = pd.concat([
    sdd2["loq"].dropna(),
    sdd1["loq"].dropna(),
], ignore_index=True)

loq_median = loq_all.median()
loq_max    = loq_all.max()

print(f"\nLOQ stats (combined, pre-merge):")
print(f"  Median LOQ : {loq_median}")
print(f"  Max LOQ    : {loq_max}")
print(f"  N with LOQ : {len(loq_all):,}  |  N missing LOQ: {(pd.concat([sdd2['loq'], sdd1['loq']]).isna()).sum():,}")

# detected  : already created (value > LOD)
# detected2 : value > median LOQ  (uniform threshold across all samples)
# detected3 : value > max LOQ     (most conservative uniform threshold)

for df in [sdd2, sdd1]:
    val = df["value"].fillna(0)
    df["detected2"] = (val > loq_median).astype("int8")
    df["detected3"] = (val > loq_max).astype("int8")


LOQ stats (combined, pre-merge):
  Median LOQ : 10.0
  Max LOQ    : 100.0
  N with LOQ : 14,527  |  N missing LOQ: 0


In [13]:
# -- 6c. PRODUCT HARMONISATION ---------------------------------------------
PRODUCT_MAP = {
    # Wheat
    "wheat":                    "wheat",
    "common wheat grain":       "wheat",
    "wheat grains (feed)":      "wheat",
    "wheat and similar-":       "wheat",
    "spelt grain":              "wheat",
    "einkorn wheat grain":      "wheat",
    "emmer wheat grain":        "wheat",

    # Maize
    "maize":                    "maize",
    "maize grain":              "maize",
    "maize grain (feed)":       "maize",
    "maize and similar-":       "maize",

    # Barley
    "barley":                   "barley",
    "barley grain (feed)":      "barley",
    "barley grains":            "barley",

    # Oats
    "oat grain":                "oat",
    "oat grain (feed)":         "oat",

    # Rye
    "rye grain":                "rye",
    "rye grain (feed)":         "rye",
    "rye and similar-":         "rye",

    # Rice
    "rice grain":               "rice",
    "rice grain, long-grain":   "rice",
    "rice, husked/brown (feed)":"rice",
    "rice and similar-":        "rice",

    # Millet
    "common millet (feed)":     "millet",
    "common millet grain":      "millet",

    # Other
    "triticale grain (feed)":   "triticale",
    "quinoa grain":             "quinoa",
    "buckwheat":                "buckwheat",
    "other cereals":            "other",
}

for df in [sdd2, sdd1]:
    df["product_harmonised"] = (
        df["product"].str.lower().str.strip().map(PRODUCT_MAP).fillna("other")
    )

KEEP_CROPS = {"wheat", "maize", "barley"}
sdd2 = sdd2[sdd2["product_harmonised"].isin(KEEP_CROPS)].copy()
sdd1 = sdd1[sdd1["product_harmonised"].isin(KEEP_CROPS)].copy()

print(f"\nAfter crop filter  SDD2: {len(sdd2):,} rows   SDD1: {len(sdd1):,} rows")
print(f"\nproduct_harmonised distribution:")
print(pd.concat([sdd2, sdd1])["product_harmonised"].value_counts())


After crop filter  SDD2: 5,170 rows   SDD1: 7,073 rows

product_harmonised distribution:
product_harmonised
wheat     5287
barley    4982
maize     1974
Name: count, dtype: int64


In [14]:
# -- 6d. LINK WEATHER DATA (SDD1 ONLY) ------------------------------------------
# Join keys: samplecountry -> code2, product_harmonised -> product,
# harvest_year -> year_harvest. Weather is only linked to SDD1 (SDD2 keeps its
# own pre-linked weather columns, untouched).
weather = weather.rename(columns={
    "code2":         "samplecountry",
    "product":       "product_harmonised",
    "year_harvest":  "harvest_year",
})
weather["samplecountry"]      = weather["samplecountry"].str.lower().str.strip()
weather["product_harmonised"] = weather["product_harmonised"].str.lower().str.strip()
weather["harvest_year"]       = pd.to_numeric(weather["harvest_year"], errors="coerce")
 
WEATHER_JOIN_KEYS = ["samplecountry", "product_harmonised", "harvest_year"]
weather_cols_to_keep = [c for c in weather.columns if c not in ("code3", "most_detailed_region")]
weather = weather[weather_cols_to_keep].drop_duplicates(subset=WEATHER_JOIN_KEYS)
 
def attach_weather(df):
    before = len(df)
    merged_df = df.merge(weather, on=WEATHER_JOIN_KEYS, how="left")
    extra_cols = weather.columns.difference(WEATHER_JOIN_KEYS)
    matched = merged_df[extra_cols[0]].notna().sum() if len(extra_cols) > 0 else 0
    print(f"  Weather match: {matched:,} / {before:,} rows")
    return merged_df
 
print("\nLinking weather data to SDD1:")
sdd1 = attach_weather(sdd1)
 





Linking weather data to SDD1:
  Weather match: 7,073 / 7,073 rows


In [15]:
print("\nSDD1 columns after weather join:")
print(sdd1.columns.tolist())


SDD1 columns after weather join:
['sample_id', 'source', 'samplecountry', 'harvest_year', 'contaminant', 'product', 'product_use', 'value', 'lod', 'loq', 'detected', 'progtype', 'labaccred', 'samppoint', 'sampmethod', 'sampstrategy', 'origcountry', 'sampm', 'sampd', 'evalcode', 'restype', 'reslegallimit', 'legallimit', 'samplehash', 'sampid', 'sampanid', 'progid', 'proglegalref', 'labcountry', 'labsubsampcode', 'anportseq', 'anportsizeunit', 'sampy', 'analysisy', 'analysism', 'analysisd', 'filename', 'country_code', 'filetype', 'resid', 'paramcode', 'restypecode', 'resunit', 'measurement_nr', 'file_id', 'product_id', 'evallowlimit', 'param_id', 'productgroup', 'origcountry_id', 'code3', 'sampcountry_id', 'production_tons_x', 'year', 'corruption_index', 'human_development_index', 'control_corruption', 'government_effectiveness', 'political_stability', 'regulatory_quality', 'rule_of_law', 'voice_and_accountability', 'governance_index', 'press_freedom_index', 'democracy_index', 'polity_d

In [18]:
# -- 6E. ENNIATIN A/A1/A+A1 AND B/B1/B+B1 OVERLAP CHECK (pre-aggregation) ------
# MM's comment: "De LOQ is heel erg verschillend, dat klopt? Er is overlap in de
# Enniatins --> wat doe je daarmee? Zijn dit verschillende samples? Of zijn er
# ook samples waar en Enniatin A en A + A1 in is gemeten?"
#
# The sample_id-level check showed zero overlap, but that check happened AFTER
# aggregation, which can hide things (e.g. if sample_id isn't unique across
# source datasets, or if a join upstream silently dropped rows). Checking here,
# right after concat and before any aggregation, rules that out.

enniatin_groups = {
    "A": ["enniatin a", "enniatin a1", "enniatin a + a1"],
    "B": ["enniatin b", "enniatin b1", "enniatin b + b1"],
}

for label, contaminants in enniatin_groups.items():
    print(f"\n--- Enniatin {label} group: pre-aggregation check ---")
    subset = merged[merged["contaminant"].isin(contaminants)]

    # counts per contaminant, split by source dataset (SDD1=EFSA / SDD2=Alltech)
    print(subset.groupby(["contaminant", "source"])["sample_id"].nunique().unstack(fill_value=0))

    # sample-level overlap, same logic as before but on the un-aggregated frame
    sample_sets = {
        c: set(subset.loc[subset["contaminant"] == c, "sample_id"])
        for c in contaminants
    }
    individual, individual2, combined = contaminants
    print(f"\n{individual}: {len(sample_sets[individual])} unique sample_ids")
    print(f"{individual2}: {len(sample_sets[individual2])} unique sample_ids")
    print(f"{combined}: {len(sample_sets[combined])} unique sample_ids")
    print(f"Overlap {individual} & {combined}: {len(sample_sets[individual] & sample_sets[combined])}")
    print(f"Overlap {individual2} & {combined}: {len(sample_sets[individual2] & sample_sets[combined])}")

    # is the combined measure's sample_id set confined to a single source?
    combined_sources = subset.loc[subset["contaminant"] == combined, "source"].unique()
    individual_sources = subset.loc[subset["contaminant"] == individual, "source"].unique()
    print(f"Source(s) reporting '{combined}': {combined_sources}")
    print(f"Source(s) reporting '{individual}': {individual_sources}")

# Check: are sample_ids globally unique, or could the same physical sample
# have been assigned the same sample_id independently in SDD1 and SDD2?
dupe_check = merged.groupby("sample_id")["source"].nunique()
n_mixed = (dupe_check > 1).sum()
print(f"\nsample_ids appearing under more than one source: {n_mixed}")
if n_mixed > 0:
    print("These sample_ids are shared across SDD1 and SDD2 - inspect before trusting")
    print("the sample-level aggregation:")
    print(dupe_check[dupe_check > 1].head(20))


--- Enniatin A group: pre-aggregation check ---
source           alltech  efsa
contaminant                   
enniatin a             0  1124
enniatin a + a1      517     0
enniatin a1            0  1158

enniatin a: 1124 unique sample_ids
enniatin a1: 1158 unique sample_ids
enniatin a + a1: 517 unique sample_ids
Overlap enniatin a & enniatin a + a1: 0
Overlap enniatin a1 & enniatin a + a1: 0
Source(s) reporting 'enniatin a + a1': <StringArray>
['alltech']
Length: 1, dtype: str
Source(s) reporting 'enniatin a': <StringArray>
['efsa']
Length: 1, dtype: str

--- Enniatin B group: pre-aggregation check ---
source           alltech  efsa
contaminant                   
enniatin b             0  1164
enniatin b + b1      517     0
enniatin b1            0  1161

enniatin b: 1164 unique sample_ids
enniatin b1: 1161 unique sample_ids
enniatin b + b1: 517 unique sample_ids
Overlap enniatin b & enniatin b + b1: 0
Overlap enniatin b1 & enniatin b + b1: 0
Source(s) reporting 'enniatin b + b1': <St

In [16]:

#  -- 7. CONCATENATE & SAVE -----------------------------------------------------
merged = pd.concat([sdd2, sdd1], ignore_index=True, sort=False)
merged = merged.loc[:, ~merged.columns.duplicated(keep="first")]

print(f"\nMerged shape: {merged.shape}")
print(f"\ndetected:\n{merged['detected'].value_counts(dropna=False)}")
print(f"\nsource:\n{merged['source'].value_counts()}")
print(f"\ncountry:\n{merged['samplecountry'].value_counts()}")
print(f"\nyear:\n{merged['harvest_year'].value_counts().sort_index()}")

print(f"\ndetected2:\n{merged['detected2'].value_counts(dropna=False)}")
print(f"\ndetected3:\n{merged['detected3'].value_counts(dropna=False)}")

# -- GUARDRAIL: every weather column must be within its domain-plausible range --
def check_all_weather_cols(data, label):
    weather_cols = [c for c in data.columns if c.startswith("month_") or c.startswith("week_")]
    all_good = True
    for col in weather_cols:
        rng = expected_range(col)
        if rng is None:
            continue
        vals = pd.to_numeric(data[col], errors="coerce")
        n_bad = ((vals < rng[0]) | (vals > rng[1])).sum()
        if n_bad > 0:
            print(f"  [FAIL] {col}: {n_bad} values outside {rng}")
            all_good = False
    if all_good:
        print(f"  All weather columns in {label} within expected ranges.")
    return all_good

print("\nFinal guardrail check:")
assert check_all_weather_cols(merged, "merged"), "Fix before saving - see FAIL lines above."

merged.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved -> {OUTPUT_PATH}  ({len(merged.columns)} columns)")

print("Sample counts per crop per year:")
print(merged.groupby(["product_harmonised", "harvest_year"]).size().unstack().to_string())

print("\ndetected2 rate per crop per year:")
print(merged.groupby(["product_harmonised", "harvest_year"])["detected2"].mean().round(3).unstack().to_string())

print("\nContaminants per crop:")
print(merged.groupby(["product_harmonised", "contaminant"]).size().unstack(fill_value=0).to_string())


Merged shape: (12243, 447)

detected:
detected
0    8932
1    3311
Name: count, dtype: int64

source:
source
efsa       7073
alltech    5170
Name: count, dtype: int64

country:
samplecountry
dk    5610
nl    3127
de    1911
cz    1595
Name: count, dtype: int64

year:
harvest_year
2019.0    1505
2020.0    2618
2021.0    2861
2022.0    2860
2023.0    2399
Name: count, dtype: int64

detected2:
detected2
0    10331
1     1912
Name: count, dtype: int64

detected3:
detected3
0    11677
1      566
Name: count, dtype: int64

Final guardrail check:
  All weather columns in merged within expected ranges.

Saved -> merged_mycotoxins.csv  (447 columns)
Sample counts per crop per year:
harvest_year        2019.0  2020.0  2021.0  2022.0  2023.0
product_harmonised                                        
barley                 533     884    1198    1376     991
maize                  387     675     338     433     141
wheat                  585    1059    1325    1051    1267

detected2 rate per cr

In [17]:
#import pandas as pd

#merged = pd.read_csv("merged_mycotoxins.csv", low_memory=False)

print("Sample counts per crop per year:")
print(merged.groupby(["product_harmonised", "harvest_year"]).size().unstack().to_string())

print("\ndetected2 rate per crop per year:")
print(merged.groupby(["product_harmonised", "harvest_year"])["detected2"].mean().round(3).unstack().to_string())

print("\nContaminants per crop:")
print(merged.groupby(["product_harmonised", "contaminant"]).size().unstack(fill_value=0).to_string())

Sample counts per crop per year:
harvest_year        2019.0  2020.0  2021.0  2022.0  2023.0
product_harmonised                                        
barley                 533     884    1198    1376     991
maize                  387     675     338     433     141
wheat                  585    1059    1325    1051    1267

detected2 rate per crop per year:
harvest_year        2019.0  2020.0  2021.0  2022.0  2023.0
product_harmonised                                        
barley               0.375   0.301   0.220   0.177   0.144
maize                0.059   0.053   0.101   0.113   0.028
wheat                0.156   0.161   0.120   0.128   0.074

Contaminants per crop:
contaminant         altenuene  alternariol  alternariol monomethyl ether  altertoxin i  beauvericin  enniatin a  enniatin a + a1  enniatin a1  enniatin b  enniatin b + b1  enniatin b1  ergocorninine  ergocristinine  ergocryptinine  ergometrinine  ergosinine  ergotaminine  tentoxin  tenuazonic acid
product_harmonised 